# पाठ 12 - एजेन्ट स्क्र्याचप्याडसँग च्याट इतिहास कमी गर्नुहोस्

यस नोटबुकले Microsoft Agent Framework प्रयोग गरेर लामो संवादहरूमा सन्दर्भ कसरी व्यवस्थापन गर्ने देखाउँछ। जति संवादहरू बढ्छन्, टोकन गणना बढ्छ — अन्ततः मोडेलको सन्दर्भ विण्डो भन्दा बाहिर जान्छ। हामी यसलाई **सन्दर्भ सारांशकरण ढाँचा** र **एजेन्ट स्क्र्याचप्याड** प्रयोग गरेर स्थायी स्मृतिका लागि समाधान गर्छौं।

## तपाईं के सिक्नुहुनेछ:
1. **किन सन्दर्भ व्यवस्थापन महत्त्वपूर्ण छ**: टोकन सीमाहरू र सन्दर्भ विन्डोहरू बुझ्न
2. **सन्दर्भ-सचेत एजेन्टहरू**: आफ्नै संवाद सन्दर्भ व्यवस्थापन गर्ने एजेन्टहरू बनाउने
3. **सन्दर्भ सारांशकरण ढाँचा**: संवाद इतिहासलाई सङ्क्षेप गर्न उपकरणहरू प्रयोग गर्ने
4. **एजेन्ट स्क्र्याचप्याड**: सन्दर्भ कमी हुँदा पनि बच्ने स्थायी स्मृति

## पूर्वआवश्यकताहरू:
- वातावरण भेरिएबलहरू कन्फिगर गरिएको Azure OpenAI सेटअप
- अघिल्ला पाठहरूबाट आधारभूत एजेन्ट अवधारणाहरूको समझ


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## किन सन्दर्भ व्यवस्थापन महत्त्वपूर्ण छ

प्रत्येक LLM सँग सीमित **सन्दर्भ विन्डो** हुन्छ — अधिकतम संख्या टोकनहरूले जुन यसले एकल अनुरोधमा प्रक्रिया गर्न सक्छ। बहु-टर्न संवाद अघि बढ्दै जाँदा:

- **टोकन गणना प्रत्येक प्रयोगकर्ता सन्देश र सहायक प्रतिक्रियासँग रैखिक रूपमा बढ्दछ।**
- **प्रॉम्प्ट टोकनहरूले लागतमा प्रभुत्व जमाउँछन्** किनकि सम्पूर्ण इतिहास प्रत्येक चरणमा पुनः पठाइन्छ।
- अन्ततः संवाद **सन्दर्भ विन्डो भन्दा बढी हुन्छ** र मोडेलले या त ट्रन्केट गर्छ वा त्रुटि गर्छ।

### सन्दर्भ व्यवस्थापनका लागि रणनीतिहरू

| रणनीति | यो कसरी कार्य गर्छ | व्यापार-बन्दी |
|---|---|---|
| **ट्रन्केसन** | पुराना सन्देशहरू हटाउनुहोस् | पुरानो सन्दर्भ हराउँछ |
| **सङ्क्षेप** | पुराना सन्देशहरूलाई सङ्क्षेपमा रूपान्तरण गर्नुहोस् | केही विवरण हराउँछ, तर मुख्य बुँदाहरू कायम रहन्छ |
| **स्क्र्याचप्याड / बाह्य स्मृति** | संवाद बाहिर मुख्य तथ्यहरू भण्डार गर्नुहोस् | उपकरण कलहरू आवश्यक हुन्छ, तर कुनै पनि कमी निकाल्न टिक्छ |

यस नोटबुकमा हामी **सङ्क्षेप** लाई **स्क्र्याचप्याड उपकरण** सँग मिलाएर प्रयोग गर्छौं ताकि एजेन्ट संवाद इतिहास सङ्क्षिप्त हुँदा पनि निरन्तरतामा रहन सकोस्।


## सन्दर्भ-चेतन एजेन्ट सिर्जना गर्दै


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## लामो संवाद अनुकरण गर्दै

सन्दर्भ कसरी संचित हुन्छ भनेर हेर्नको लागि हामी बहु-पटकको संवादमार्फत जान्छौं। एजेन्टले मुख्य विवरणहरू (रुचिहरू, बजेट, यात्रा मिति) टर्नहरू बीचमा राख्नुपर्छ र निरन्तरता प्रदर्शन गर्नुपर्छ।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ध्यान दिनुहोस् कि एजेन्टले पहिलेका पालाहरूको सन्दर्भ कायम राख्छ — यो जापान, सुशी, मन्दिरहरू, फोटोग्राफी, $3000 बजेट, एक्लै यात्रा, र अप्रिलको समयसीमा बारे जान्दछ। सानो संवादमा यो राम्रोसँग काम गर्दछ, तर जति बढ्छ संवाद, पूर्ण इतिहास पुन: पठाउन महँगो हुन्छ।

सन्दर्भ संचय कसरी हुन्छ भनेर बुझ्न थप पालाहरू सहितको संवाद जारी राखौं:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## सन्दर्भ संक्षेपण ढाँचा

वार्तालाप बढेसँगै, हामीले जम्मा गरिएको सन्दर्भलाई सञ्‍क्षिप्त ढाँचामा संक्षेप गर्न **सारांश उपकरण** प्रयोग गर्न सक्छौं। एजेन्टले यो उपकरणलाई महत्वपूर्ण प्राथमिकताहरू रेकर्ड गर्न कल गर्छ ताकि पुराना सन्देशहरू हटाइए पनि, आवश्यक जानकारी सुरक्षित रहोस्।

यो ढाँचा अधिक जटिल इतिहास कटौतीको लागि आधार हो:
1. एजेन्टले वार्तालापबाट मुख्य तथ्यहरू पहिचान गर्छ
2. यसले तिनलाई कायम राख्न सारांश उपकरणलाई कल गर्छ
3. पुराना सन्देशहरू सुरक्षित रूपमा हटाउन सकिन्छ किनभने सारांशले महत्वपूर्ण कुरा समेट्छ

तल हामी `summarize_preferences` उपकरण परिभाषित गर्छौं जसलाई एजेण्टले सिकेको कुराको सञ्‍क्षिप्त सारांश रेकर्ड गर्न कल गर्न सक्छ।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework प्रयोग गरेर लामो अवधिको एजेन्ट संवादहरूमा सन्दर्भ कसरी व्यवस्थापन गर्ने सिक्नुभयो:

### मुख्य अवधारणाहरू
- **सन्दर्भ विन्डो सीमित हुन्छन्** — संवाद इतिहासमा प्रत्येक टोकनले खर्च लाग्छ र सीमा तर्फ गणना हुन्छ।
- **सारांश उपकरणहरूले** एजेन्टलाई संचित सन्दर्भलाई सङ्क्षिप्त सारहरूमा सङ्कुचित गर्न दिन्छन्, जसले टोकन प्रयोग घटाउँछ र आवश्यक जानकारी जोगाउँछ।
- **एजेन्ट स्क्र्याचप्याडहरूले** निरन्तर बाह्य स्मृति प्रदान गर्छन् जुन कुनै पनि संवाद घटाव पछि पनि बाँचिरहन्छ।

### तपाईंले के बनाउनु भयो
- एउटा **सन्दर्भ-चेतना एजेन्ट** जसले बहु-चरण संवादहरूमा निरन्तरता कायम गर्छ
- एउटा **सारांश उपकरण** (`summarize_preferences`) जसले प्रयोगकर्ताका मुख्य विवरणहरू सङ्कुचित स्वरूपमा रेकर्ड गर्छ
- एउटा **बहु-चरण संवाद** जसले सन्दर्भ संरक्षण र परिवर्तन व्यवस्थापन देखाउँछ

### वास्तविक संसारका अनुप्रयोगहरू
- **ग्राहक सेवा बोटहरू**: लामो समर्थन बैठकहरूमा प्राथमिकताहरू सम्झन्छन्
- **व्यक्तिगत सहायकहरू**: सन्दर्भ फेरि व्याख्या नगरी जारी परियोजनाहरू ट्र्याक गर्छन्
- **शैक्षिक ट्यूटरहरू**: धेरै अन्तरक्रियाहरूमा विद्यार्थी प्रगति कायम राख्छन्

### अर्को कदमहरू
- फाइल-आधारित स्थायित्वसहित पूर्ण स्क्र्याचप्याड उपकरण कार्यान्वयन गर्नुहोस्
- सारांश पछि स्वचालित इतिहास कटौती थप्नुहोस्
- सेमेन्टिक स्मृति खोजका लागि भेक्टर डेटाबेसहरूसँग संयोजन गर्नुहोस्
- पूरा सन्दर्भसहित दिनहरू पछि संवाद पुनः सुरू गर्न सक्ने एजेन्टहरू बनाउनुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकृति**:
यो दस्तावेज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताका लागि प्रयासरत छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुनसक्छन्। मूल दस्तावेज यसको स्थानीय भाषामा नै अधिकारिक स्रोत मानिनु पर्छ। महत्त्वपूर्ण जानकारीको लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलतफहमी वा गलत व्याख्याबाट हामी जिम्मेवार हुँदैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
